In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random

In [ ]:
import sys
from pathlib import Path

notebook_dir = Path.cwd()
root_dir = notebook_dir.parent.parent
sys.path.append(str(root_dir))

root_dir

In [ ]:
from benchmark.configs import Configs

In [ ]:
data_dir = Path.cwd().parent.parent.parent.resolve() / "data"

dataset = "download"

img_in_dir = data_dir / f"{dataset}/images"
img_out_dir = data_dir / f"images/{dataset}"

csv_out_dir = data_dir / "metadata"
metadata_path = csv_out_dir / f"{dataset}_metadata.csv"

pred_out_dir = data_dir / f"predictions/{dataset}"

In [ ]:
img_out_dir.mkdir(parents=True, exist_ok=True)
csv_out_dir.mkdir(parents=True, exist_ok=True)
pred_out_dir.mkdir(exist_ok=True, parents=True)

In [ ]:
def convert_to_B02_B03_B04(path: str | Path, img_out_dir: str | Path, show: bool = False):
    metadata = []
    path = Path(path)
    if path.is_dir():
        # Process directory
        im_files = [f for ext in ("tif", "tiff", "png", "jpg") for f in path.rglob(f"*.{ext}")]
        for im_path in im_files:
            metadata.append(convert_to_B02_B03_B04(im_path, img_out_dir, show))
        return metadata
    else:
        # Process a single image
        filename = path.stem
        bgr = cv2.imread(str(path))
        height, width = bgr.shape[:2]

        chip_out_dir = Path(img_out_dir) / filename
        chip_out_dir.mkdir(parents=True, exist_ok=True)
        row = {
            "chip_id": filename,
            "location": "E0.0-N0.0",
            "datetime": "2020/02/02",
            "x_start": 0,
            "x_end": width,
            "y_start": 0,
            "y_end": height,
            "B02_path": f"{str(chip_out_dir)}/B02.tif",
            "B03_path": f"{str(chip_out_dir)}/B03.tif",
            "B04_path": f"{str(chip_out_dir)}/B04.tif",
        }

        b = bgr[:, :, 0]
        g = bgr[:, :, 1]
        r = bgr[:, :, 2]

        cv2.imwrite(row["B02_path"], b)
        cv2.imwrite(row["B03_path"], g)
        cv2.imwrite(row["B04_path"], r)

        if show:
            plt.figure(figsize=(12, 6))

            plt.subplot(1, 2, 1)
            plt.imshow(bgr)
            plt.title("Original RGB Image", fontsize=14)
            plt.axis('off')

            plt.subplot(1, 2, 2)
            plt.imshow(nir, cmap='hot')
            plt.title("Generated NIR Image (835nm)", fontsize=14)
            plt.axis('off')

            plt.tight_layout()
            plt.show()

        return row

In [ ]:
metadata = convert_to_B02_B03_B04(img_in_dir, img_out_dir)
pd.DataFrame(metadata).to_csv(metadata_path, index=False)
metadata[0]

In [ ]:
from benchmark.inference import simple

config = Configs.balanced()
config.batch_size = 4
config.use_tta = True
model_weights_path = Path(config.model_name) / Path("assets/cloud_model.pt")
simple(model_weights_path, pd.read_csv(metadata_path), pred_out_dir, config)

In [ ]:
from PIL import Image
import xarray
import xrspatial.multispectral as ms

def get_xarray(filepath):
    """Put images in xarray.DataArray format"""
    im_arr = np.array(Image.open(filepath))
    return xarray.DataArray(im_arr, dims=["y", "x"])

def true_color_img(B02_path, B03_path, B04_path):
    """Given the path to the directory of Sentinel-2 chip feature images,
    plots the true color image"""
    red = get_xarray(B04_path)
    green = get_xarray(B03_path)
    blue = get_xarray(B02_path)

    return ms.true_color(r=red, g=green, b=blue, c=5, th=0.3)

In [ ]:
files = list(pred_out_dir.glob("*.tif"))

pred_files = random.sample(files, len(files))
for pred_file in pred_files:
    B02_file = img_out_dir / pred_file.stem / "B02.tif"
    B03_file = img_out_dir / pred_file.stem / "B03.tif"
    B04_file = img_out_dir / pred_file.stem / "B04.tif"

    fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    if B02_file.exists():
        ax[0].imshow(true_color_img(B02_file, B03_file, B04_file))
        ax[0].set_title(f"True color image for Chip {pred_file.stem}")
    if pred_file.exists():
        pred = Image.open(pred_file)
        ax[2].imshow(pred)
        ax[2].set_title(f"Predicted Chip {pred_file.stem} label")
    plt.tight_layout()
    plt.show()